# TreeSearchPlayer.estimate_opponent() をオーバーライドして、相手の見えていない情報（未公開の技構成）を推定する方法を示す

estimate_opponent()が呼ばれる条件・既定の挙動（何もしない場合はfallback()に
委譲される）は docs/api/README.md の TreeSearchPlayer「オーバーライド可能なフック」節
（03_tree_search_ai.py参照）を参照。

Google Colabで開いた場合は、まず次のセルで `jpoke` をインストールする
（ローカルで `pip install jpoke` 済みなら不要）。

In [ ]:
%pip install -q jpoke

In [ ]:
from __future__ import annotations

from jpoke import Battle, Player
from jpoke.players.tree_search_player import TreeSearchPlayer

In [ ]:
class OpponentGuessingPlayer(TreeSearchPlayer):
    """相手の未公開の技を「みずでっぽう」1本と仮定して探索するAI（estimate_opponent()の拡張例）。"""

    def estimate_opponent(self, battle: Battle, opponent: Player) -> None:
        # 実戦では対戦データベースや使用率統計等から推定するが、ここでは固定の
        # 推測技で最小例を示す。opponent.active.moves が空（未公開）のときだけ
        # 書き込む
        opponent_active = battle.get_active(opponent)
        if not opponent_active.moves:
            opponent_active.set_moves(["みずでっぽう"])

In [ ]:
ai_player = OpponentGuessingPlayer("GuessingAI", max_plies=1, max_nodes=50)
ai_player.add_pokemon("リザードン", move_names=["かえんほうしゃ", "じしん"])

opponent_player = Player("Opponent")
opponent_player.add_pokemon("カメックス", move_names=["みずでっぽう"])

In [ ]:
battle = Battle(ai_player, opponent_player, seed=1)
battle.start()

In [ ]:
# 1ターン目: 相手の技は未公開。estimate_opponent が推定を書き込むため
# fallback() には委譲されず、推定した相手の技も踏まえた探索が行われる
# （nodes_expanded > 0 になる。何も推定しない既定実装のままだと0のまま）
battle.step()
print(f"1ターン目に展開したノード数: {ai_player.nodes_expanded}")
battle.print_logs()

試してみよう: estimate_opponent() の中身を消して既定実装（何もしない）に
戻すと、1ターン目は探索されず nodes_expanded が0のままになることを確認できる